# P19 — Entrenar modelos de lenguaje grandes con cómputo óptimo

## 1. Título y paper

**Paper:** *Training Compute-Optimal Large Language Models*  
**Autoría:** Jordan Hoffmann, Sebastian Borgeaud, Arthur Mensch, y otros (DeepMind)  
**Año y venue:** 2022 · arXiv:2203.15556 · NeurIPS 2022  
**Nivel:** L4 · **Motor:** `scaling_laws`  
**Ficha completa:** [`P19_scaling_laws`](../../papers/foundational/P19_scaling_laws/README.md)

**Hito:** Corrige la carrera por el tamaño: a cómputo fijo, los modelos de la época estaban infraentrenados en datos.

- [arXiv:2203.15556](https://arxiv.org/abs/2203.15556)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Tras GPT-3 la industria escalaba parámetros asumiendo que era la variable dominante, sin medir el reparto óptimo entre parámetros y tokens a cómputo constante.
2. Ejecutar una implementación mínima de la propuesta: Ajustar empíricamente L(N, D) y resolver el reparto que minimiza la pérdida bajo la restricción C = 6ND.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P10
- Kaplan et al. (2020), las leyes que este trabajo corrige


## 4. Intuición

Tienes un presupuesto fijo de cómputo. Puedes gastarlo en un modelo enorme entrenado con pocos datos, o en uno mediano entrenado con muchos. No son equivalentes, y durante años la industria eligió sistemáticamente mal.


## 5. Concepto mínimo

```text
Forma paramétrica ajustada empíricamente:
    L(N, D) = E + A/N^α + B/D^β

Restricción de presupuesto (aproximación estándar):
    C ≈ 6·N·D          N = parámetros, D = tokens, C = FLOPs de entrenamiento

El problema es: minimizar L(N, D) sujeto a 6ND = C
```

`E` es el error irreducible (la entropía del lenguaje). Los otros dos términos son lo que se compra con parámetros y con datos respectivamente.


## 6. Código explicado

El motor evalúa varias asignaciones **con el mismo cómputo** y encuentra el óptimo. Las constantes son didácticas: la forma es lo transferible.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('scaling_laws', seed=7)['result']
print('forma:', r['forma_parametrica'])
show(r['constantes'])
print('\npresupuesto fijo:', r['presupuesto_flops'], 'FLOPs\n')
for c in r['candidatos_a_igual_computo']:
    print(f"N={c['N_parametros']} · D={c['D_tokens']} · {c['tokens_por_parametro']:>8} tok/param · L={c['perdida']}")

## 7. Predicción antes de ejecutar

1. ¿La pérdida será igual en todas las asignaciones del mismo cómputo, o habrá un mínimo claro?
2. ¿El óptimo estará en «el modelo más grande posible»?
3. Si duplicas el presupuesto, ¿qué duplicarías: N, D, o ambos a medias?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
E, A, B, alpha, beta = 1.69, 400.0, 400.0, 0.34, 0.28
def L(N, D):
    return E + A / N ** alpha + B / D ** beta

for factor in (1, 2, 4, 8):
    C = factor * 6 * 70e9 * 1.4e12
    mejor = min(((N, C / (6 * N)) for N in (10 ** e * m for e in range(10, 13) for m in (1, 2, 5))),
                key=lambda nd: L(*nd))
    print(f'presupuesto x{factor}: N*={mejor[0]:.1e} D*={mejor[1]:.1e} '
          f'tok/param={mejor[1]/mejor[0]:.0f} L={L(*mejor):.4f}')

## 9. Salida interpretable

Al crecer el presupuesto, **N y D crecen a la vez**: ninguna de las dos absorbe todo el incremento. Ese es el resultado que reordenó la industria — antes se subía N y se dejaba D casi fijo, gastando cómputo en parámetros que nunca llegaban a entrenarse bien.


## 10. Comentario pedagógico

Las constantes de este notebook **son didácticas**, no las del paper: sirven para que la curva tenga la forma correcta, no para citar un número. Si necesitas los valores ajustados, están en el artículo. Confundir ambas cosas es exactamente lo que este eje entrena a no hacer.


## 11. Error o anti-patrón deliberado

Anti-patrón: extrapolar la ley fuera del rango donde se ajustó, o usarla para predecir *capacidades* en vez de pérdida.


In [ ]:
print('L(N,D) predice PERDIDA DE PREENTRENAMIENTO.')
print('No predice: si el modelo razonará, si alucinará menos, ni si servirá para tu tarea.')
print('Tampoco cubre el coste de INFERENCIA, que hoy suele dominar el coste total.')

## 12. Corrección

Lo que la ley sí autoriza a decir, y lo que no:


In [ ]:
afirmaciones = {
    'valido': ['a computo fijo existe un reparto optimo entre N y D',
               'los modelos de 2020-2021 estaban infraentrenados en datos'],
    'no_valido': ['un modelo con menor perdida es mejor para mi tarea',
                  'la ley se extrapola varios ordenes de magnitud',
                  'el modelo optimo para entrenar es el optimo para servir'],
}
show(afirmaciones)

## 13. Desafío guiado

Un modelo servido a millones de usuarios se entrena una vez y se ejecuta siempre. Calcula cuándo conviene un modelo más pequeño que el óptimo de entrenamiento.


In [ ]:
coste_entrenamiento = lambda N, D: 6 * N * D
coste_inferencia = lambda N, peticiones, tokens: 2 * N * peticiones * tokens
N_opt, D_opt = 7e10, 1.4e12
for peticiones in (1e6, 1e9, 1e12):
    e = coste_entrenamiento(N_opt, D_opt)
    i = coste_inferencia(N_opt, peticiones, 500)
    print(f'{peticiones:.0e} peticiones → entrenamiento {e:.2e} vs inferencia {i:.2e} '
          f"({'inferencia domina' if i > e else 'entrenamiento domina'})")

## 14. Desafío autónomo

Ajusta tu propia ley de escalado entrenando modelos diminutos (10K–10M parámetros) sobre un corpus público, con varios presupuestos. Estima α y β y comprueba si tu óptimo predicho se cumple en una configuración que no usaste para ajustar.


## 15. Evidencia de aprendizaje

Guarda la tabla de asignaciones a igual cómputo, el óptimo encontrado y la lista de lo que la ley **no** autoriza a afirmar.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P19_scaling_laws/README.md) · evaluación formal: [`assessments/papers/P19_scaling_laws.md`](../../assessments/papers/P19_scaling_laws.md)


## 16. Cierre

Ya sabemos cuánto gastar en cada cosa. La pregunta siguiente vuelve a la arquitectura: el coste cuadrático de la atención sigue ahí, y alguien tenía que atacarlo.


## 17. Conexión con el siguiente hito

- P21
- P22

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
